# The Slice Filter

*Available as of Qdrant v1.19.0*

The `slice` condition divides a collection into a fixed number of deterministic, disjoint subsets, and matches every point in one of those subsets. A point matches slice `index` of `total` if `hash(id) % total == index`, using SipHash-2-4 over the point ID bytes. For a fixed `total`, slices `0` through `total - 1` are disjoint and together cover every point in the collection.

That determinism is the whole point. Unlike [random sampling](https://qdrant.tech/documentation/search/search/index.md#random-sampling), a given slice always returns the same points, and it composes with any other filter condition.

This notebook covers three uses:

- Splitting a collection into disjoint chunks for parallel [scroll](https://qdrant.tech/documentation/concepts/points/#scroll-points), so several workers can export, migrate, or re-embed points without overlapping.
- Drawing a reproducible sample of a collection for recall evaluation or a train/test split.
- Combining `slice` with a payload filter for stratified sampling, for example a canary rollout limited to one category.


### Setup

This notebook runs against a [Qdrant Cloud](https://cloud.qdrant.io) cluster. The [free tier](https://qdrant.tech/documentation/cloud/create-cluster/) is enough, since the collection here is small.

We also use [Qdrant Cloud Inference](https://qdrant.tech/documentation/inference/cloud-inference/) to embed the sample data, so there's no local embedding model to load. Enable Cloud Inference for your cluster from the Inference tab of the Cluster Detail page in the Qdrant Cloud console, where you can also see which models are free to use, marked with a "Cost: Free" label. This tutorial uses `sentence-transformers/all-MiniLM-L6-v2`, one of the free models.

In [ ]:
!pip install -q qdrant-client

Export your cluster URL and API key, both available from the Qdrant Cloud console, then connect with `cloud_inference=True`.

In [ ]:
import os

from qdrant_client import AsyncQdrantClient, models

client = AsyncQdrantClient(
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"],
    # Without this, inference falls back to running locally.
    cloud_inference=True,
)

collection_name = "slicing-demo"
embedding_model = "sentence-transformers/all-MiniLM-L6-v2"

await client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE),
)


Upload a small catalog of product descriptions. Each point's vector is a `Document` object, embedded server-side on Qdrant Cloud.

In [4]:
import random

from qdrant_client.models import Document

random.seed(0)

products = {
    "electronics": ["wireless earbuds", "4K monitor", "mechanical keyboard", "USB-C hub", "smartwatch"],
    "books": ["science fiction novel", "cookbook", "history book", "graphic novel", "poetry collection"],
    "clothing": ["running shoes", "wool sweater", "denim jacket", "rain jacket", "cotton t-shirt"],
    "home": ["cast iron pan", "ceramic mug", "throw blanket", "desk lamp", "storage basket"],
}
categories = list(products)

points = []
for i in range(500):
    category = random.choice(categories)
    description = random.choice(products[category])
    points.append(
        models.PointStruct(
            id=i,
            payload={"category": category},
            vector=Document(text=description, model=embedding_model),
        )
    )

client.upload_points(collection_name=collection_name, points=points, wait=True)
await client.count(collection_name=collection_name)


CountResult(count=500)

## Scrolling a Single Slice

A `SliceCondition` takes an `index` and a `total`. Here we ask for slice `3` out of `8`, one eighth of the collection.

In [5]:
result, _ = await client.scroll(
    collection_name=collection_name,
    scroll_filter=models.Filter(
        must=[
            models.SliceCondition(slice=models.Slice(index=3, total=8)),
        ],
    ),
    limit=500,
    with_payload=False,
    with_vectors=False,
)

print(f"slice 3 of 8: {len(result)} points")


slice 3 of 8: 61 points


With `total: 8` and 500 points, each slice holds roughly 60 points, close to `500 / 8`.

## Concurrent Scrolling Across Workers

The main use case is splitting a full scroll into `N` independent, non-overlapping scrolls, one per worker. Each worker only needs its own `index`, and since the slices don't overlap, the requests have nothing to coordinate on and can run concurrently. We use `AsyncQdrantClient` and `asyncio.gather` here to fire all four requests at once, rather than waiting on them one at a time.

In [6]:
import asyncio


async def scroll_slice(index: int, total: int) -> list[models.Record]:
    records, _ = await client.scroll(
        collection_name=collection_name,
        scroll_filter=models.Filter(
            must=[models.SliceCondition(slice=models.Slice(index=index, total=total))],
        ),
        limit=500,
        with_payload=True,
        with_vectors=False,
    )
    return records


total_slices = 4
slices = await asyncio.gather(*(scroll_slice(i, total_slices) for i in range(total_slices)))

for i, s in enumerate(slices):
    print(f"worker {i}: {len(s)} points")

print(f"sum across workers: {sum(len(s) for s in slices)}")


worker 0: 115 points
worker 1: 143 points
worker 2: 118 points
worker 3: 124 points
sum across workers: 500


Every point appears in exactly one slice, so the counts add up to the full collection with no overlap and no gaps.

In [7]:
ids_per_slice = [{r.id for r in s} for s in slices]
overlap = set.intersection(*ids_per_slice)
union = set.union(*ids_per_slice)

count_result = await client.count(collection_name=collection_name)

print(f"overlapping IDs: {len(overlap)}")
print(f"union covers full collection: {len(union) == count_result.count}")


overlapping IDs: 0
union covers full collection: True


## Reproducible Sampling

Because the hash is stable across runs and Qdrant versions, a single slice makes a reproducible sample. Requesting slice `0` of `total: 10` always returns the same 10% of the collection, which makes it a solid choice for a recall benchmark or a train/test split that has to be repeatable.

In [8]:
sample, _ = await client.scroll(
    collection_name=collection_name,
    scroll_filter=models.Filter(
        must=[models.SliceCondition(slice=models.Slice(index=0, total=10))],
    ),
    limit=10000,
    with_payload=False,
    with_vectors=False,
)

sample_ids = {p.id for p in sample}

# Running the same query again returns the identical set of points.
sample_again, _ = await client.scroll(
    collection_name=collection_name,
    scroll_filter=models.Filter(
        must=[models.SliceCondition(slice=models.Slice(index=0, total=10))],
    ),
    limit=10000,
    with_payload=False,
    with_vectors=False,
)

print(f"sample size: {len(sample_ids)}")
print(f"identical on repeat: {sample_ids == {p.id for p in sample_again}}")


sample size: 47
identical on repeat: True


Slices with different `total` values are also correlated: slice `0` of `total: 4` is always a subset of slice `0` of `total: 2`. Growing the number of slices refines a sample instead of reshuffling it.

In [9]:
coarse, _ = await client.scroll(
    collection_name=collection_name,
    scroll_filter=models.Filter(must=[models.SliceCondition(slice=models.Slice(index=0, total=2))]),
    limit=10000,
    with_payload=False,
    with_vectors=False,
)

fine, _ = await client.scroll(
    collection_name=collection_name,
    scroll_filter=models.Filter(must=[models.SliceCondition(slice=models.Slice(index=0, total=4))]),
    limit=10000,
    with_payload=False,
    with_vectors=False,
)

fine_ids = {p.id for p in fine}
coarse_ids = {p.id for p in coarse}

print(f"slice 0/4 is a subset of slice 0/2: {fine_ids.issubset(coarse_ids)}")


slice 0/4 is a subset of slice 0/2: True


## Stratified Sampling with Payload Filters

`slice` is a normal filter condition, so it combines with any other condition. This gives a reproducible sample restricted to one category, useful for a canary rollout or for evaluating a change against a single segment of the data.

In [11]:
from qdrant_client.models import PayloadSchemaType

await client.create_payload_index(
    collection_name=collection_name,
    field_name="category",
    field_schema=PayloadSchemaType.KEYWORD,
)

stratified, _ = await client.scroll(
    collection_name=collection_name,
    scroll_filter=models.Filter(
        must=[
            models.SliceCondition(slice=models.Slice(index=0, total=5)),
            models.FieldCondition(key="category", match=models.MatchValue(value="electronics")),
        ],
    ),
    limit=10000,
    with_payload=True,
    with_vectors=False,
)

print(f"electronics points in slice 0/5: {len(stratified)}")
print(f"all match the category: {all(p.payload['category'] == 'electronics' for p in stratified)}")


electronics points in slice 0/5: 28
all match the category: True
